## Probar con DOC2
## Se reemplaza la imagen de curva por la imagen de OBRAS y se usa la imagen de CRUCE de color naranja

In [27]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import cv2

# Define la carpeta donde estan las imagenes PNG sin fondo.
carpeta='img/sinfondo'

# Busca todas las imagenes .png dentro de la carpeta y las ordena.
imagenes=sorted(glob.glob(os.path.join(carpeta,'*.png')))

# Filtra archivos para evitar el cruce viejo y dejar la version nueva (crucenar).
imagenes=[r for r in imagenes if ('cruce' not in os.path.basename(r).lower()) or ('crucenar' in os.path.basename(r).lower())]

# Rangos HSV para color azul (limites inferior y superior).
azul_b=np.array([85,70,60],np.uint8)
azul_a=np.array([135,255,255],np.uint8)

# Rangos HSV para color amarillo.
amarillo_b=np.array([18,70,70],np.uint8)
amarillo_a=np.array([40,255,255],np.uint8)

# Rangos HSV para rojo en dos zonas (rojo bajo y rojo alto en HSV).
rojo_b1=np.array([0,100,70],np.uint8)
rojo_a1=np.array([10,255,255],np.uint8)
rojo_b2=np.array([170,100,70],np.uint8)
rojo_a2=np.array([180,255,255],np.uint8)

# Kernel para limpieza morfologica (open/close).
kernel=np.ones((3,3),np.uint8)

# Area minima para ignorar ruido pequeno.
min_area=120

# Recorre cada imagen encontrada.
for ruta in imagenes:
    # Lee la imagen manteniendo alpha si existe.
    img=cv2.imread(ruta,cv2.IMREAD_UNCHANGED)

    # Si no se pudo leer, avisa y pasa a la siguiente.
    if img is None:
        print('No se pudo leer:',ruta)
        continue

    # Si trae canal alpha, separa BGR y alpha.
    if len(img.shape)==3 and img.shape[2]==4:
        bgr=img[:,:,:3].copy()
        alpha=img[:,:,3]
    else:
        # Si no trae alpha, usa imagen normal.
        bgr=img.copy()
        alpha=None

    # Estandariza tamano para resultados consistentes.
    bgr=cv2.resize(bgr,(500,500))

    # Suaviza la imagen para reducir ruido antes de segmentar color.
    frame_s=cv2.GaussianBlur(bgr,(7,7),0)

    # Convierte de BGR a HSV para detectar colores con mas estabilidad.
    hsv=cv2.cvtColor(frame_s,cv2.COLOR_BGR2HSV)

    # Crea mascara binaria de azul.
    mask_azul=cv2.inRange(hsv,azul_b,azul_a)

    # Crea mascara binaria de amarillo.
    mask_amarillo=cv2.inRange(hsv,amarillo_b,amarillo_a)

    # Une las dos mascaras de rojo para cubrir todo el rojo en HSV.
    mask_rojo=cv2.bitwise_or(cv2.inRange(hsv,rojo_b1,rojo_a1),cv2.inRange(hsv,rojo_b2,rojo_a2))

    # Si habia alpha, limita deteccion solo al objeto visible (no fondo transparente).
    if alpha is not None:
        alpha=cv2.resize(alpha,(500,500))
        _,mask_obj=cv2.threshold(alpha,1,255,cv2.THRESH_BINARY)
        mask_azul=cv2.bitwise_and(mask_azul,mask_obj)
        mask_amarillo=cv2.bitwise_and(mask_amarillo,mask_obj)
        mask_rojo=cv2.bitwise_and(mask_rojo,mask_obj)

    # Limpieza morfologica en azul para quitar puntos y cerrar huecos.
    mask_azul=cv2.morphologyEx(mask_azul,cv2.MORPH_OPEN,kernel)
    mask_azul=cv2.morphologyEx(mask_azul,cv2.MORPH_CLOSE,kernel)

    # Limpieza morfologica en amarillo.
    mask_amarillo=cv2.morphologyEx(mask_amarillo,cv2.MORPH_OPEN,kernel)
    mask_amarillo=cv2.morphologyEx(mask_amarillo,cv2.MORPH_CLOSE,kernel)

    # Limpieza morfologica en rojo.
    mask_rojo=cv2.morphologyEx(mask_rojo,cv2.MORPH_OPEN,kernel)
    mask_rojo=cv2.morphologyEx(mask_rojo,cv2.MORPH_CLOSE,kernel)

    # Busca contornos en cada mascara.
    cont_azul,_=cv2.findContours(mask_azul,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    cont_amarillo,_=cv2.findContours(mask_amarillo,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    cont_rojo,_=cv2.findContours(mask_rojo,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)

    # Copia para dibujar resultados.
    salida=bgr.copy()

    # Dibuja contornos de objetos azules en color rojo si superan area minima.
    for c in cont_azul:
        area=cv2.contourArea(c)
        if area>min_area:
            cv2.drawContours(salida,[c],-1,(0,0,255),2)

    # Dibuja contornos de objetos rojos en color amarillo si superan area minima.
    for c in cont_rojo:
        area=cv2.contourArea(c)
        if area>min_area:
            cv2.drawContours(salida,[c],-1,(0,255,255),2)

    # Dibuja contornos de objetos amarillos en color azul si superan area minima.
    for c in cont_amarillo:
        area=cv2.contourArea(c)
        if area>min_area:
            cv2.drawContours(salida,[c],-1,(255,0,0),2)

    # Nombre del archivo actual para titulos e impresion.
    nombre=os.path.basename(ruta)

    # Convierte imagenes a RGB para mostrar correctamente en matplotlib.
    bgr_rgb=cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB)
    salida_rgb=cv2.cvtColor(salida,cv2.COLOR_BGR2RGB)

    # Crea figura de visualizacion.
    plt.figure(figsize=(12,4))

    # Panel 1: imagen original.
    plt.subplot(1,3,1)
    plt.imshow(bgr_rgb,vmin=0,vmax=255)
    plt.title(f'Original: {nombre}')
    plt.xticks([])
    plt.yticks([])

    # Panel 2: mascara combinada para ver pixeles detectados.
    plt.subplot(1,3,2)
    mezcla=cv2.add(mask_rojo,cv2.add(mask_amarillo,mask_azul))
    plt.imshow(mezcla,vmin=0,vmax=255,cmap='gray')
    plt.title('Mascara combinada RGB')
    plt.xticks([])
    plt.yticks([])

    # Panel 3: contornos finales dibujados.
    plt.subplot(1,3,3)
    plt.imshow(salida_rgb,vmin=0,vmax=255)
    plt.title('Contornos por color')
    plt.xticks([])
    plt.yticks([])

    # Muestra la figura completa.
    plt.show()

    # Imprime resumen de cuantos contornos detecto por color.
    print(nombre,'-> azul:',len(cont_azul),'rojo:',len(cont_rojo),'amarillo:',len(cont_amarillo))

In [28]:
import numpy as np
import cv2

# Abre la camara por defecto (indice 0).
cam=cv2.VideoCapture(0)

# Si no abre, muestra error. Si abre, inicia todo el pipeline.
if not cam.isOpened():
    print('No se pudo abrir la camara')
else:
    # ===== 1) RANGOS HSV POR COLOR =====
    # Azul: usado para detectar AUTOBUS.
    azul_b=np.array([85,70,60],np.uint8)
    azul_a=np.array([135,255,255],np.uint8)

    # Naranja: doble rango para cubrir tonos cercanos a H=0 y H=179.
    # Ajustado a tonos como: 9F6764, 9F645F, 996263, A86B66.
    naranja_b1=np.array([0,85,130],np.uint8)
    naranja_a1=np.array([12,170,215],np.uint8)
    naranja_b2=np.array([175,85,130],np.uint8)
    naranja_a2=np.array([179,170,215],np.uint8)

    # Rojo (limite de velocidad): calibrado a tonos como
    # 854A52, 92514F, 8C4E55, 84535A (rojo oscuro/tenue).
    rojo_b1=np.array([0,75,90],np.uint8)
    rojo_a1=np.array([8,190,200],np.uint8)
    rojo_b2=np.array([172,75,90],np.uint8)
    rojo_a2=np.array([180,190,200],np.uint8)

    # Negro: para detectar texto interno "20" y "km/h".
    negro_b=np.array([0,0,0],np.uint8)
    negro_a=np.array([180,120,95],np.uint8)

    # Verde: usado para detectar DESTINO.
    verde_b=np.array([35,60,60],np.uint8)
    verde_a=np.array([90,255,255],np.uint8)

    # Kernel base para operaciones morfologicas.
    kernel=np.ones((3,3),np.uint8)

    # Area minima para ignorar ruido pequeno.
    min_area=450

    # Bucle principal de video en tiempo real.
    while True:
        # Lee un frame de camara.
        ret,frame=cam.read()

        # Si no llega frame, termina.
        if not ret:
            break

        # Obtiene alto y ancho del frame actual.
        h_frame,w_frame=frame.shape[:2]

        # ===== 2) ROI CENTRAL =====
        # Define una zona central para detectar solo ahi y reducir ruido.
        x1=int(w_frame*0.25)
        y1=int(h_frame*0.20)
        x2=int(w_frame*0.75)
        y2=int(h_frame*0.80)

        # Copia solo la region de interes.
        roi=frame[y1:y2,x1:x2].copy()

        # Suaviza para reducir ruido de sensor.
        frame_s=cv2.GaussianBlur(roi,(7,7),0)

        # Convierte a HSV para segmentar color con mas estabilidad.
        hsv=cv2.cvtColor(frame_s,cv2.COLOR_BGR2HSV)

        # ===== 3) MASCARAS POR COLOR =====
        mask_azul=cv2.inRange(hsv,azul_b,azul_a)
        mask_naranja=cv2.bitwise_or(cv2.inRange(hsv,naranja_b1,naranja_a1),cv2.inRange(hsv,naranja_b2,naranja_a2))
        mask_rojo=cv2.bitwise_or(cv2.inRange(hsv,rojo_b1,rojo_a1),cv2.inRange(hsv,rojo_b2,rojo_a2))
        mask_negro=cv2.inRange(hsv,negro_b,negro_a)
        mask_verde=cv2.inRange(hsv,verde_b,verde_a)

        # ===== 4) LIMPIEZA MORFOLOGICA =====
        # Azul
        mask_azul=cv2.morphologyEx(mask_azul,cv2.MORPH_OPEN,kernel)
        mask_azul=cv2.morphologyEx(mask_azul,cv2.MORPH_CLOSE,kernel)

        # Naranja
        mask_naranja=cv2.morphologyEx(mask_naranja,cv2.MORPH_OPEN,kernel)
        mask_naranja=cv2.morphologyEx(mask_naranja,cv2.MORPH_CLOSE,kernel)

        # Rojo
        mask_rojo=cv2.morphologyEx(mask_rojo,cv2.MORPH_OPEN,kernel)
        mask_rojo=cv2.morphologyEx(mask_rojo,cv2.MORPH_CLOSE,kernel)

        # Negro
        mask_negro=cv2.morphologyEx(mask_negro,cv2.MORPH_OPEN,kernel)
        mask_negro=cv2.morphologyEx(mask_negro,cv2.MORPH_CLOSE,kernel)

        # Verde
        mask_verde=cv2.morphologyEx(mask_verde,cv2.MORPH_OPEN,kernel)
        mask_verde=cv2.morphologyEx(mask_verde,cv2.MORPH_CLOSE,kernel)

        # Une rectangulos verdes cercanos para no detectar varios DESTINO separados.
        mask_verde_union=cv2.dilate(mask_verde,np.ones((9,9),np.uint8),iterations=1)
        mask_verde_union=cv2.morphologyEx(mask_verde_union,cv2.MORPH_CLOSE,np.ones((11,11),np.uint8))

        # ===== 5) CONTORNOS =====
        cont_azul,_=cv2.findContours(mask_azul,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
        cont_naranja,_=cv2.findContours(mask_naranja,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
        cont_rojo,_=cv2.findContours(mask_rojo,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
        cont_verde,_=cv2.findContours(mask_verde_union,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)

        # ===== 6) AZUL -> AUTOBUS =====
        # Se elige solo el contorno azul mas grande para evitar duplicados.
        mejor_azul=None
        area_azul_max=0
        for c in cont_azul:
            area=cv2.contourArea(c)
            if area>min_area and area>area_azul_max:
                area_azul_max=area
                mejor_azul=c

        # Dibuja y etiqueta AUTOBUS si encontro contorno valido.
        if mejor_azul is not None:
            x,y,w,h=cv2.boundingRect(mejor_azul)
            precision=min(99,int(60+(area_azul_max/1800)))
            cv2.drawContours(roi,[mejor_azul],-1,(0,0,255),3)
            cv2.putText(roi,'AUTOBUS '+str(precision)+'%',(x,y-8),cv2.FONT_HERSHEY_SIMPLEX,0.62,(0,0,255),2)

        # ===== 7) ROJO -> 20KMH / ALTO / CEDA EL PASO / OBRAS =====
        for c in cont_rojo:
            area=cv2.contourArea(c)
            if area<=min_area:
                continue

            per=cv2.arcLength(c,True)
            if per<=0:
                continue

            # Aproxima poligono para contar lados.
            ap=cv2.approxPolyDP(c,0.02*per,True)
            lados=len(ap)

            # Caja y metricas geometricas.
            x,y,w,h=cv2.boundingRect(c)
            asp=w/float(h) if h>0 else 0.0
            circ=(4*np.pi*area)/(per*per)

            # Fill circular: cuanto llena el contorno su circulo envolvente.
            _,radio=cv2.minEnclosingCircle(c)
            fill_circ=0.0
            if radio>0:
                fill_circ=area/(np.pi*radio*radio)

            # Mide presencia de negro en el interior para identificar "20" y "km/h".
            xi=max(x+int(0.22*w),0)
            yi=max(y+int(0.22*h),0)
            xf=min(x+int(0.78*w),roi.shape[1])
            yf=min(y+int(0.78*h),roi.shape[0])
            black_ratio=0.0
            if xf>xi and yf>yi:
                roi_neg=mask_negro[yi:yf,xi:xf]
                area_in=float((yf-yi)*(xf-xi))
                if area_in>0:
                    black_ratio=cv2.countNonZero(roi_neg)/area_in

            etiqueta=''
            precision=0

            # 20KMH primero: circulo rojo con texto negro interno.
            if circ>0.78 and 0.82<asp<1.20 and fill_circ>0.80 and lados>=6 and black_ratio>0.012:
                etiqueta='20KMH'
                precision=min(99,int(72+(circ*16)+(fill_circ*8)+(black_ratio*260)))

            # ALTO: octagono aproximado y normalmente sin negro interno fuerte.
            elif 6<=lados<=10 and 0.65<asp<1.35 and 0.42<circ<0.88 and fill_circ<0.94 and black_ratio<0.10:
                etiqueta='ALTO'
                p_lados=100-abs(8-lados)*8
                p_circ=int(min(100,max(0,(0.88-circ)*220)))
                precision=max(70,min(99,int((p_lados+p_circ)/2)))

            # Triangulos rojos: CEDA (invertido) y OBRAS (normal).
            elif lados==3:
                pts=ap.reshape(-1,2)
                ys=np.sort(pts[:,1])

                # CEDA: base arriba y punta abajo.
                if abs(int(ys[0])-int(ys[1])) < (0.20*h) and abs(int(ys[2])-int(ys[1])) > (0.25*h):
                    etiqueta='CEDA EL PASO'
                    precision=90

                # OBRAS: base abajo y punta arriba.
                elif abs(int(ys[2])-int(ys[1])) < (0.20*h) and abs(int(ys[1])-int(ys[0])) > (0.25*h):
                    etiqueta='OBRAS'
                    precision=90

            # Dibuja contorno rojo detectado.
            cv2.drawContours(roi,[c],-1,(0,255,255),3)

            # Escribe etiqueta solo si se clasifico algo.
            if etiqueta!='':
                cv2.putText(roi,etiqueta+' '+str(precision)+'%',(x,y-8),cv2.FONT_HERSHEY_SIMPLEX,0.58,(0,255,255),2)

        # ===== 8) NARANJA (ROMBO) -> CRUCE =====
        mejor_naranja=None
        area_naranja_max=0

        # Elige el rombo naranja mas grande.
        for c in cont_naranja:
            area=cv2.contourArea(c)
            if area<=min_area:
                continue
            per=cv2.arcLength(c,True)
            if per<=0:
                continue
            ap=cv2.approxPolyDP(c,0.02*per,True)
            lados=len(ap)
            x,y,w,h=cv2.boundingRect(c)
            asp=w/float(h) if h>0 else 0.0
            if lados==4 and 0.70<asp<1.35 and area>area_naranja_max:
                area_naranja_max=area
                mejor_naranja=c

        # Dibuja CRUCE cuando encuentra rombo naranja valido.
        if mejor_naranja is not None:
            x,y,w,h=cv2.boundingRect(mejor_naranja)
            precision=min(99,int(72+(area_naranja_max/2300)))
            cv2.drawContours(roi,[mejor_naranja],-1,(255,0,0),3)
            cv2.putText(roi,'CRUCE '+str(precision)+'%',(x,y-8),cv2.FONT_HERSHEY_SIMPLEX,0.58,(255,0,0),2)

        # ===== 9) VERDE -> DESTINO =====
        mejor_verde=None
        area_verde_max=0

        # Toma el contorno verde agrupado mas grande.
        for c in cont_verde:
            area=cv2.contourArea(c)
            if area>min_area and area>area_verde_max:
                area_verde_max=area
                mejor_verde=c

        # Dibuja caja y etiqueta DESTINO.
        if mejor_verde is not None:
            x,y,w,h=cv2.boundingRect(mejor_verde)
            m=8
            x=max(0,x-m)
            y=max(0,y-m)
            w=min(roi.shape[1]-x,w+2*m)
            h=min(roi.shape[0]-y,h+2*m)
            precision=min(99,int(65+(area_verde_max/2200)))
            cv2.rectangle(roi,(x,y),(x+w,y+h),(0,255,0),3)
            cv2.putText(roi,'DESTINO '+str(precision)+'%',(x,y-8),cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,255,0),2)

        # ===== 10) SALIDA EN PANTALLA =====
        # Pega ROI procesada al frame completo.
        frame[y1:y2,x1:x2]=roi

        # Dibuja el cuadro guia de deteccion.
        cv2.rectangle(frame,(x1,y1),(x2,y2),(255,255,0),2)

        # Muestra ventana final.
        cv2.imshow('Deteccion por camara - ProyectoExitoso',frame)

        # Presiona tecla 0 para salir del bucle.
        if cv2.waitKey(1)&0xFF==ord('0'):
            break

# Libera la camara y cierra ventanas al terminar.
cam.release()
cv2.destroyAllWindows()